# 05 — Data Validation
## Nexora Supply Chain Analytics Platform

**Business objective:** confirm that the final analytics-ready dataset is reliable, internally consistent, suitable for MySQL loading, and safe for Tableau analysis.

**Preferred input**
- `dataco_supply_chain_analytics_ready.csv`

**Fallback input**
- `dataco_supply_chain_cleaned.csv`

**Outputs**
- `data_validation_report.csv`
- `data_validation_summary.xlsx`
- `data_quality_issues.csv`
- `data_readiness_score.csv`

**Validation scope**
- Schema
- Missing values
- Duplicate records
- Primary-key integrity
- Data types
- Date consistency
- Numeric ranges
- Business rules
- Feature consistency
- Referential readiness
- MySQL and Tableau readiness

## 1. Validation principles

1. Validation should detect errors without silently changing the data.
2. Every check must have a clear expected result.
3. Critical failures must block MySQL loading.
4. Warning-level issues may be accepted if documented.
5. The final readiness score must be traceable to individual checks.

In [1]:
import importlib.util
import subprocess
import sys

required = ["pandas", "numpy", "openpyxl"]
for package in required:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Load the final dataset

In [2]:
FILE_CANDIDATES = [
    "/content/nexora_outputs/dataco_supply_chain_analytics_ready.csv",
    "/content/dataco_supply_chain_analytics_ready.csv",
    "/content/nexora_outputs/dataco_supply_chain_cleaned.csv",
    "/content/dataco_supply_chain_cleaned.csv",
    "/mnt/data/dataco_supply_chain_analytics_ready.csv",
]

def resolve_input_file(candidates):
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate

    try:
        from google.colab import files
        print("Upload dataco_supply_chain_analytics_ready.csv")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file was uploaded.")
        return next(iter(uploaded.keys()))
    except ImportError as exc:
        raise FileNotFoundError(
            "Dataset not found. Update FILE_CANDIDATES with the correct path."
        ) from exc

def read_csv_robust(path):
    for encoding in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise RuntimeError("Unable to detect CSV encoding.")

FILE_PATH = resolve_input_file(FILE_CANDIDATES)
df = read_csv_robust(FILE_PATH)

print(f"Source file : {FILE_PATH}")
print(f"Dataset size: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head())

Upload dataco_supply_chain_analytics_ready.csv


Saving dataco_supply_chain_cleaned.csv to dataco_supply_chain_cleaned.csv
Source file : dataco_supply_chain_cleaned.csv
Dataset size: 180,519 rows × 59 columns


,payment_type,days_for_shipping,days_for_shipment,order_profit,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_id,customer_segment,customer_state,customer_zipcode,department_id,department_name,latitude,longitude,market,order_city,order_country,order_customer_id,order_date,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_state,order_status,order_zipcode,product_card_id,product_category_id,product_name,product_price,product_status,shipping_date,shipping_mode,is_late_delivery,profit_margin_pct,gross_sales_before_discount,profitability_status,order_year,order_quarter,order_month,order_month_name,order_week,order_day,order_day_name,order_date_key,shipping_date_key
0,CASH,2,4,88.7900,239.9800,Advance Shipping,0,43,Camping & Hiking,Hickory,Ee. Uu.,11599,Consumer,Nc,"28,601.0000",7,Fan Shop,35.7767,-81.3626,LATAM,Mexico City,México,11599,2015-01-01 00:00:00,1,957,60.0000,0.2000,1,299.9800,0.3700,1,299.9800,239.9800,88.7900,Central America,Distrito Federal,CLOSED,NaN,957,43,Diamondback Women's Serene Classic Comfort Bi,299.9800,0,2015-01-03 00:00:00,Standard Class,0,29.5986,359.9800,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150103
1,PAYMENT,3,4,91.1800,193.9900,Advance Shipping,0,48,Water Sports,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.0000",7,Fan Shop,41.8327,-87.9805,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,1073,6.0000,0.0300,2,199.9900,0.4700,1,199.9900,193.9900,91.1800,South America,Risaralda,PENDING_PAYMENT,NaN,1073,48,Pelican Sunstream 100 Kayak,199.9900,0,2015-01-04 00:21:00,Standard Class,0,45.5923,205.9900,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104
2,PAYMENT,3,4,68.2500,227.5000,Advance Shipping,0,24,Women's Apparel,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.0000",5,Golf,41.8327,-87.9805,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,502,22.5000,0.0900,3,50.0000,0.3000,5,250.0000,227.5000,68.2500,South America,Risaralda,PENDING_PAYMENT,NaN,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.0000,0,2015-01-04 00:21:00,Standard Class,0,27.3000,272.5000,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104
3,PAYMENT,3,4,36.4700,107.8900,Advance Shipping,0,18,Men's Footwear,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.0000",4,Apparel,41.8327,-87.9805,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,403,22.1000,0.1700,4,129.9900,0.3400,1,129.9900,107.8900,36.4700,South America,Risaralda,PENDING_PAYMENT,NaN,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.9900,0,2015-01-04 00:21:00,Standard Class,0,28.0560,152.0900,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104
4,CASH,5,4,4.1000,40.9800,Late Delivery,1,40,Accessories,San Antonio,Ee. Uu.,8827,Home Office,Tx,"78,240.0000",6,Outdoors,29.5200,-98.6374,LATAM,Dos Quebradas,Colombia,8827,2015-01-01 01:03:00,4,897,9.0000,0.1800,5,24.9900,0.1000,2,49.9800,40.9800,4.1000,South America,Risaralda,CLOSED,NaN,897,40,Team Golf New England Patriots Putter Grip,24.9900,0,2015-01-06 01:03:00,Standard Class,1,8.2033,58.9800,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150106


## 3. Standardize analytical data types

In [3]:
date_candidates = [
    "order_date",
    "shipping_date",
    "customer_first_order_date",
    "customer_last_order_date",
]

numeric_candidates = [
    "order_item_id", "order_id", "customer_id", "order_customer_id",
    "product_card_id", "category_id", "department_id",
    "sales", "order_profit", "order_profit_per_order",
    "order_item_discount", "order_item_discount_rate",
    "discount_rate_pct", "profit_margin_pct",
    "order_item_product_price", "order_item_quantity",
    "days_for_shipping_real", "days_for_shipment_scheduled",
    "shipping_delay_days", "late_delivery_risk",
    "is_late_delivery", "is_on_time_delivery",
    "latitude", "longitude"
]

for col in [c for c in date_candidates if c in df.columns]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

for col in [c for c in numeric_candidates if c in df.columns]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Data types standardized for validation.")

Data types standardized for validation.


## 4. Validation framework

In [4]:
validation_results = []
issue_rows = []

SEVERITY_WEIGHT = {
    "Critical": 5,
    "High": 4,
    "Medium": 3,
    "Low": 2,
    "Info": 1,
}

def add_validation(
    category,
    check_name,
    severity,
    passed,
    observed,
    expected,
    affected_rows=0,
    notes="",
):
    validation_results.append({
        "category": category,
        "check_name": check_name,
        "severity": severity,
        "status": "PASS" if bool(passed) else "FAIL",
        "observed": observed,
        "expected": expected,
        "affected_rows": int(affected_rows),
        "affected_pct": round((affected_rows / len(df) * 100) if len(df) else 0, 4),
        "notes": notes,
        "weight": SEVERITY_WEIGHT[severity],
    })

def add_issue(category, issue_type, column, row_index, value, notes=""):
    issue_rows.append({
        "category": category,
        "issue_type": issue_type,
        "column": column,
        "row_index": row_index,
        "value": value,
        "notes": notes,
    })

print("Validation framework initialized.")

Validation framework initialized.


## 5. Dataset-level checks

In [5]:
add_validation(
    "Dataset",
    "Dataset is not empty",
    "Critical",
    len(df) > 0,
    len(df),
    "> 0 rows",
    0 if len(df) > 0 else len(df),
)

add_validation(
    "Dataset",
    "Dataset contains expected large volume",
    "High",
    len(df) >= 180_000,
    len(df),
    "At least 180,000 rows",
    0 if len(df) >= 180_000 else len(df),
    "This check detects partial imports such as only a few hundred rows.",
)

add_validation(
    "Dataset",
    "Column names are unique",
    "Critical",
    len(df.columns) == len(set(df.columns)),
    len(df.columns),
    "All column names unique",
    len(df.columns) - len(set(df.columns)),
)

invalid_column_names = [
    c for c in df.columns if not re.fullmatch(r"[a-z][a-z0-9_]*", c)
]
add_validation(
    "Schema",
    "Column names use snake_case",
    "Medium",
    len(invalid_column_names) == 0,
    invalid_column_names,
    "All names match [a-z][a-z0-9_]*",
    len(invalid_column_names),
)

## 6. Required-column validation

In [7]:
required_columns = [
    "order_item_id",
    "order_id",
    "sales",
    "order_date",
    "shipping_date",
    "days_for_shipping_real",
    "days_for_shipment_scheduled",
]

missing_required = [c for c in required_columns if c not in df.columns]

add_validation(
    "Schema",
    "Required columns are available",
    "Critical",
    len(missing_required) == 0,
    missing_required,
    "No required column missing",
    len(missing_required),
)

print("Missing required columns:", missing_required)

Missing required columns: ['days_for_shipping_real', 'days_for_shipment_scheduled']


## 7. Missing-value validation

In [8]:
missing_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [round(df[c].isna().mean() * 100, 4) for c in df.columns],
}).sort_values("missing_pct", ascending=False)

critical_non_null = [
    c for c in [
        "order_item_id", "order_id", "sales",
        "order_date", "shipping_date"
    ] if c in df.columns
]

for col in critical_non_null:
    count = int(df[col].isna().sum())
    add_validation(
        "Completeness",
        f"{col} has no missing values",
        "Critical" if col in ["order_item_id", "order_id"] else "High",
        count == 0,
        count,
        0,
        count,
    )

high_missing_columns = missing_summary.loc[
    missing_summary["missing_pct"] > 50, "column"
].tolist()

add_validation(
    "Completeness",
    "No column exceeds 50% missing values",
    "Medium",
    len(high_missing_columns) == 0,
    high_missing_columns,
    "No column above 50% missing",
    len(high_missing_columns),
)

display(missing_summary.head(20))

,column,missing_count,missing_pct
38,order_zipcode,155679,86.2397
14,customer_zipcode,3,0.0017
1,days_for_shipping,0,0.0000
3,order_profit,0,0.0000
2,days_for_shipment,0,0.0000
4,sales_per_customer,0,0.0000
5,delivery_status,0,0.0000
7,category_id,0,0.0000
6,late_delivery_risk,0,0.0000
9,customer_city,0,0.0000


## 8. Duplicate and key-integrity checks

In [9]:
exact_duplicates = int(df.duplicated().sum())
add_validation(
    "Uniqueness",
    "No exact duplicate rows",
    "High",
    exact_duplicates == 0,
    exact_duplicates,
    0,
    exact_duplicates,
)

if "order_item_id" in df.columns:
    duplicate_item_ids = int(df["order_item_id"].duplicated(keep=False).sum())
    add_validation(
        "Primary Key",
        "order_item_id is unique",
        "Critical",
        duplicate_item_ids == 0,
        duplicate_item_ids,
        0,
        duplicate_item_ids,
    )

    missing_item_ids = int(df["order_item_id"].isna().sum())
    add_validation(
        "Primary Key",
        "order_item_id is not null",
        "Critical",
        missing_item_ids == 0,
        missing_item_ids,
        0,
        missing_item_ids,
    )

if {"order_id", "order_item_id"}.issubset(df.columns):
    items_without_order = int(df["order_id"].isna().sum())
    add_validation(
        "Referential Readiness",
        "Every order item has an order_id",
        "Critical",
        items_without_order == 0,
        items_without_order,
        0,
        items_without_order,
    )

## 9. Date validation

In [6]:
for col in [c for c in ["order_date", "shipping_date"] if c in df.columns]:
    invalid_dates = int(df[col].isna().sum())
    add_validation(
        "Date",
        f"{col} contains valid parsed dates",
        "High",
        invalid_dates == 0,
        invalid_dates,
        0,
        invalid_dates,
    )

if {"order_date", "shipping_date"}.issubset(df.columns):
    shipping_before_order = int((df["shipping_date"] < df["order_date"]).sum())
    add_validation(
        "Business Rule",
        "Shipping date is not earlier than order date",
        "Critical",
        shipping_before_order == 0,
        shipping_before_order,
        0,
        shipping_before_order,
    )

    for idx in df.index[df["shipping_date"] < df["order_date"]][:100]:
        add_issue(
            "Date",
            "Shipping before order",
            "shipping_date",
            int(idx),
            df.at[idx, "shipping_date"],
            f"order_date={df.at[idx, 'order_date']}",
        )

if "order_date" in df.columns:
    year_min = df["order_date"].dt.year.min()
    year_max = df["order_date"].dt.year.max()
    add_validation(
        "Date",
        "Order years fall within plausible dataset period",
        "Medium",
        pd.notna(year_min) and year_min >= 2000 and year_max <= 2035,
        f"{year_min}–{year_max}",
        "2000–2035",
        0 if pd.notna(year_min) and year_min >= 2000 and year_max <= 2035 else len(df),
    )

## 10. Numeric-range validation

In [7]:
non_negative_columns = [
    "sales",
    "order_item_discount",
    "order_item_product_price",
    "order_item_quantity",
    "days_for_shipping_real",
    "days_for_shipment_scheduled",
]

for col in [c for c in non_negative_columns if c in df.columns]:
    negative_count = int((df[col] < 0).sum())
    add_validation(
        "Numeric Range",
        f"{col} contains no negative values",
        "High",
        negative_count == 0,
        negative_count,
        0,
        negative_count,
    )

if "order_item_quantity" in df.columns:
    invalid_quantity = int((df["order_item_quantity"] <= 0).sum())
    add_validation(
        "Business Rule",
        "Order-item quantity is greater than zero",
        "High",
        invalid_quantity == 0,
        invalid_quantity,
        0,
        invalid_quantity,
    )

if "discount_rate_pct" in df.columns:
    invalid_discount = int(
        ((df["discount_rate_pct"] < 0) | (df["discount_rate_pct"] > 100)).sum()
    )
    add_validation(
        "Numeric Range",
        "Discount rate is between 0% and 100%",
        "High",
        invalid_discount == 0,
        invalid_discount,
        0,
        invalid_discount,
    )

if "order_item_discount_rate" in df.columns:
    rate = df["order_item_discount_rate"]
    invalid_rate = int(((rate < 0) | (rate > 1)).sum())
    add_validation(
        "Numeric Range",
        "Raw discount rate is between 0 and 1",
        "High",
        invalid_rate == 0,
        invalid_rate,
        0,
        invalid_rate,
    )

if "latitude" in df.columns:
    invalid_latitude = int(((df["latitude"] < -90) | (df["latitude"] > 90)).sum())
    add_validation(
        "Geography",
        "Latitude is within valid range",
        "High",
        invalid_latitude == 0,
        invalid_latitude,
        0,
        invalid_latitude,
    )

if "longitude" in df.columns:
    invalid_longitude = int(((df["longitude"] < -180) | (df["longitude"] > 180)).sum())
    add_validation(
        "Geography",
        "Longitude is within valid range",
        "High",
        invalid_longitude == 0,
        invalid_longitude,
        0,
        invalid_longitude,
    )

## 11. Shipping and delivery-rule validation

In [8]:
if {"days_for_shipping_real", "days_for_shipment_scheduled"}.issubset(df.columns):
    calculated_delay = (
        df["days_for_shipping_real"] - df["days_for_shipment_scheduled"]
    )

    if "shipping_delay_days" in df.columns:
        mismatch = int(
            ~np.isclose(
                df["shipping_delay_days"].fillna(999999),
                calculated_delay.fillna(999999),
                equal_nan=True,
            )
        .sum()
        )
        add_validation(
            "Feature Consistency",
            "shipping_delay_days matches source columns",
            "Critical",
            mismatch == 0,
            mismatch,
            0,
            mismatch,
        )

    if "is_late_delivery" in df.columns:
        expected_late = calculated_delay.gt(0).astype(int)
        actual_late = df["is_late_delivery"].fillna(-1).astype(int)
        mismatch = int((expected_late != actual_late).sum())

        add_validation(
            "Feature Consistency",
            "is_late_delivery matches shipping delay",
            "Critical",
            mismatch == 0,
            mismatch,
            0,
            mismatch,
        )

    if "is_on_time_delivery" in df.columns:
        expected_on_time = calculated_delay.le(0).astype(int)
        actual_on_time = df["is_on_time_delivery"].fillna(-1).astype(int)
        mismatch = int((expected_on_time != actual_on_time).sum())

        add_validation(
            "Feature Consistency",
            "is_on_time_delivery matches shipping delay",
            "High",
            mismatch == 0,
            mismatch,
            0,
            mismatch,
        )

if "is_late_delivery" in df.columns:
    invalid_binary = int((~df["is_late_delivery"].isin([0, 1])).sum())
    add_validation(
        "Data Type",
        "is_late_delivery is binary",
        "High",
        invalid_binary == 0,
        invalid_binary,
        0,
        invalid_binary,
    )

if "late_delivery_risk" in df.columns:
    invalid_binary = int((~df["late_delivery_risk"].isin([0, 1])).sum())
    add_validation(
        "Data Type",
        "late_delivery_risk is binary",
        "High",
        invalid_binary == 0,
        invalid_binary,
        0,
        invalid_binary,
    )

## 12. Financial consistency validation

In [9]:
profit_col = (
    "order_profit_per_order"
    if "order_profit_per_order" in df.columns
    else "order_profit" if "order_profit" in df.columns else None
)

if {"sales", "order_item_discount", "gross_sales_before_discount"}.issubset(df.columns):
    expected_gross = df["sales"] + df["order_item_discount"]
    mismatch = int(
        (~np.isclose(
            df["gross_sales_before_discount"].fillna(999999),
            expected_gross.fillna(999999),
            atol=0.01,
            equal_nan=True,
        )).sum()
    )
    add_validation(
        "Feature Consistency",
        "Gross sales equals sales plus discount",
        "High",
        mismatch == 0,
        mismatch,
        0,
        mismatch,
    )

if profit_col and {"sales", "profit_margin_pct"}.issubset(df.columns):
    expected_margin = np.where(
        df["sales"].ne(0),
        df[profit_col] / df["sales"] * 100,
        np.nan,
    )
    valid_mask = df["sales"].ne(0) & df[profit_col].notna()
    mismatch = int(
        (
            ~np.isclose(
                df.loc[valid_mask, "profit_margin_pct"],
                expected_margin[valid_mask],
                atol=0.01,
                equal_nan=True,
            )
        ).sum()
    )
    add_validation(
        "Feature Consistency",
        "Profit margin matches sales and profit",
        "High",
        mismatch == 0,
        mismatch,
        0,
        mismatch,
    )

if profit_col and "profitability_status" in df.columns:
    expected_status = np.select(
        [
            df[profit_col] > 0,
            df[profit_col] == 0,
            df[profit_col] < 0,
        ],
        ["Profitable", "Break Even", "Loss"],
        default="Unknown",
    )

    mismatch = int(
        (
            df["profitability_status"].astype(str)
            != pd.Series(expected_status, index=df.index).astype(str)
        ).sum()
    )

    add_validation(
        "Feature Consistency",
        "Profitability status matches profit value",
        "High",
        mismatch == 0,
        mismatch,
        0,
        mismatch,
    )

## 13. Aggregate-feature consistency

In [10]:
if {"order_id", "sales", "order_total_sales"}.issubset(df.columns):
    recalculated = df.groupby("order_id")["sales"].transform("sum")
    mismatch = int(
        (~np.isclose(
            df["order_total_sales"].fillna(999999),
            recalculated.fillna(999999),
            atol=0.01,
            equal_nan=True,
        )).sum()
    )
    add_validation(
        "Aggregate Feature",
        "order_total_sales is consistent within each order",
        "Critical",
        mismatch == 0,
        mismatch,
        0,
        mismatch,
    )

if profit_col and {"order_id", "order_total_profit"}.issubset(df.columns):
    recalculated = df.groupby("order_id")[profit_col].transform("sum")
    mismatch = int(
        (~np.isclose(
            df["order_total_profit"].fillna(999999),
            recalculated.fillna(999999),
            atol=0.01,
            equal_nan=True,
        )).sum()
    )
    add_validation(
        "Aggregate Feature",
        "order_total_profit is consistent within each order",
        "High",
        mismatch == 0,
        mismatch,
        0,
        mismatch,
    )

if {"order_id", "order_item_count"}.issubset(df.columns):
    recalculated = df.groupby("order_id")["order_id"].transform("size")
    mismatch = int((df["order_item_count"] != recalculated).sum())
    add_validation(
        "Aggregate Feature",
        "order_item_count is consistent within each order",
        "High",
        mismatch == 0,
        mismatch,
        0,
        mismatch,
    )

## 14. Categorical-domain validation

In [11]:
known_domains = {
    "delivery_performance": {"Late", "On Schedule", "Early", "Unknown"},
    "profitability_status": {"Profitable", "Break Even", "Loss", "Unknown"},
    "operational_risk_segment": {"Late and Loss", "Late Only", "Loss Only", "Healthy"},
}

for col, allowed_values in known_domains.items():
    if col in df.columns:
        observed_values = set(df[col].dropna().astype(str).unique())
        unexpected = sorted(observed_values - allowed_values)
        add_validation(
            "Categorical Domain",
            f"{col} contains only approved values",
            "Medium",
            len(unexpected) == 0,
            unexpected,
            sorted(allowed_values),
            len(df[df[col].astype(str).isin(unexpected)]) if unexpected else 0,
        )

## 15. Privacy and sensitive-field validation

In [12]:
sensitive_columns = {
    "customer_email",
    "customer_password",
    "customer_fname",
    "customer_lname",
    "customer_street",
}

sensitive_present = sorted(sensitive_columns.intersection(df.columns))

add_validation(
    "Privacy",
    "Direct customer PII has been removed",
    "Critical",
    len(sensitive_present) == 0,
    sensitive_present,
    "No direct PII columns",
    len(sensitive_present),
    "Customer identifiers may remain as surrogate analytical keys.",
)

## 16. MySQL readiness checks

In [13]:
mysql_problem_columns = []

for col in df.columns:
    if df[col].dtype == "object":
        max_len = int(df[col].dropna().astype(str).str.len().max() or 0)
        if max_len > 65535:
            mysql_problem_columns.append((col, max_len))

add_validation(
    "MySQL Readiness",
    "Text columns fit practical MySQL limits",
    "Medium",
    len(mysql_problem_columns) == 0,
    mysql_problem_columns,
    "No text value above 65,535 characters",
    len(mysql_problem_columns),
)

object_cols = df.select_dtypes(include=["object"]).columns.tolist()
null_char_columns = []
for col in object_cols:
    has_null_char = df[col].dropna().astype(str).str.contains("\x00", regex=False).any()
    if has_null_char:
        null_char_columns.append(col)

add_validation(
    "MySQL Readiness",
    "Text columns contain no null characters",
    "High",
    len(null_char_columns) == 0,
    null_char_columns,
    "No null characters",
    len(null_char_columns),
)

## 17. Tableau readiness checks

In [14]:
duplicate_semantic_fields = []

semantic_groups = {
    "profit": [c for c in ["order_profit", "order_profit_per_order"] if c in df.columns],
    "customer": [c for c in ["customer_id", "order_customer_id"] if c in df.columns],
    "discount_rate": [c for c in ["discount_rate_pct", "order_item_discount_rate"] if c in df.columns],
}

for semantic_name, columns in semantic_groups.items():
    if len(columns) > 1:
        duplicate_semantic_fields.append({
            "semantic_name": semantic_name,
            "columns": columns,
        })

add_validation(
    "Tableau Readiness",
    "Potentially duplicate semantic fields are documented",
    "Low",
    True,
    duplicate_semantic_fields,
    "Review field naming before dashboard publication",
    0,
    "This is informational and does not block publication.",
)

all_null_columns = [c for c in df.columns if df[c].isna().all()]
add_validation(
    "Tableau Readiness",
    "No completely empty columns",
    "Medium",
    len(all_null_columns) == 0,
    all_null_columns,
    "No all-null columns",
    len(all_null_columns),
)

constant_columns = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
add_validation(
    "Tableau Readiness",
    "Constant columns are identified",
    "Low",
    True,
    constant_columns,
    "Review before dashboard use",
    0,
    "Constant columns may be removed from the published data source.",
)

## 18. Build validation report

In [15]:
validation_report = pd.DataFrame(validation_results)

validation_report["weighted_pass"] = np.where(
    validation_report["status"].eq("PASS"),
    validation_report["weight"],
    0,
)

validation_report = validation_report[
    [
        "category", "check_name", "severity", "status",
        "observed", "expected", "affected_rows",
        "affected_pct", "notes", "weight", "weighted_pass"
    ]
]

display(validation_report)

,category,check_name,severity,status,observed,expected,affected_rows,affected_pct,notes,weight,weighted_pass
0,Dataset,Dataset is not empty,Critical,PASS,180519,> 0 rows,0,0.0000,,5,5
1,Dataset,Dataset contains expected large volume,High,PASS,180519,"At least 180,000 rows",0,0.0000,This check detects partial imports such as onl...,4,4
2,Dataset,Column names are unique,Critical,PASS,59,All column names unique,0,0.0000,,5,5
3,Schema,Column names use snake_case,Medium,PASS,[],All names match [a-z][a-z0-9_]*,0,0.0000,,3,3
4,Date,order_date contains valid parsed dates,High,PASS,0,0,0,0.0000,,4,4
5,Date,shipping_date contains valid parsed dates,High,PASS,0,0,0,0.0000,,4,4
6,Business Rule,Shipping date is not earlier than order date,Critical,PASS,0,0,0,0.0000,,5,5
7,Date,Order years fall within plausible dataset period,Medium,PASS,2015–2018,2000–2035,0,0.0000,,3,3
8,Numeric Range,sales contains no negative values,High,PASS,0,0,0,0.0000,,4,4
9,Numeric Range,order_item_discount contains no negative values,High,PASS,0,0,0,0.0000,,4,4


## 19. Calculate readiness score

In [16]:
total_weight = validation_report["weight"].sum()
passed_weight = validation_report["weighted_pass"].sum()
readiness_score = round((passed_weight / total_weight * 100) if total_weight else 0, 2)

critical_failures = validation_report[
    (validation_report["severity"] == "Critical")
    & (validation_report["status"] == "FAIL")
]

high_failures = validation_report[
    (validation_report["severity"] == "High")
    & (validation_report["status"] == "FAIL")
]

if len(critical_failures) > 0:
    readiness_status = "NOT READY"
elif readiness_score >= 95 and len(high_failures) == 0:
    readiness_status = "READY"
elif readiness_score >= 85:
    readiness_status = "READY WITH WARNINGS"
else:
    readiness_status = "NEEDS IMPROVEMENT"

readiness_summary = pd.DataFrame({
    "metric": [
        "dataset_rows",
        "dataset_columns",
        "validation_checks",
        "passed_checks",
        "failed_checks",
        "critical_failures",
        "high_failures",
        "readiness_score",
        "readiness_status",
    ],
    "value": [
        len(df),
        df.shape[1],
        len(validation_report),
        int(validation_report["status"].eq("PASS").sum()),
        int(validation_report["status"].eq("FAIL").sum()),
        len(critical_failures),
        len(high_failures),
        readiness_score,
        readiness_status,
    ],
})

display(readiness_summary)

print(f"Readiness score : {readiness_score:.2f}%")
print(f"Readiness status: {readiness_status}")

,metric,value
0,dataset_rows,180519
1,dataset_columns,59
2,validation_checks,28
3,passed_checks,28
4,failed_checks,0
5,critical_failures,0
6,high_failures,0
7,readiness_score,100.0000
8,readiness_status,READY


Readiness score : 100.00%
Readiness status: READY


## 20. Failed-check summary

In [17]:
failed_checks = validation_report[
    validation_report["status"] == "FAIL"
].sort_values(
    ["weight", "affected_rows"],
    ascending=[False, False],
)

if len(failed_checks):
    display(failed_checks)
else:
    print("No validation failures detected.")

No validation failures detected.


## 21. Dataset quality profile

In [22]:
quality_profile = pd.DataFrame({
    "column": df.columns,
    "data_type": [str(df[c].dtype) for c in df.columns],
    "row_count": len(df),
    "non_null_count": [int(df[c].notna().sum()) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [round(df[c].isna().mean() * 100, 4) for c in df.columns],
    "unique_count": [int(df[c].nunique(dropna=True)) for c in df.columns],
})

display(quality_profile.head(25))

,column,data_type,row_count,non_null_count,missing_count,missing_pct,unique_count
0,payment_type,object,180519,180519,0,0.0000,4
1,days_for_shipping,int64,180519,180519,0,0.0000,7
2,days_for_shipment,int64,180519,180519,0,0.0000,4
3,order_profit,float64,180519,180519,0,0.0000,21998
4,sales_per_customer,float64,180519,180519,0,0.0000,2927
5,delivery_status,object,180519,180519,0,0.0000,4
6,late_delivery_risk,int64,180519,180519,0,0.0000,2
7,category_id,int64,180519,180519,0,0.0000,51
8,category_name,object,180519,180519,0,0.0000,50
9,customer_city,object,180519,180519,0,0.0000,563


## 22. Collect representative issue records

In [18]:
issue_samples = []

for _, row in failed_checks.iterrows():
    issue_samples.append({
        "category": row["category"],
        "issue_type": row["check_name"],
        "column": "",
        "row_index": "",
        "value": row["observed"],
        "notes": row["notes"],
    })

issue_samples.extend(issue_rows)
data_quality_issues = pd.DataFrame(issue_samples)

if len(data_quality_issues):
    display(data_quality_issues.head(100))
else:
    data_quality_issues = pd.DataFrame(
        columns=["category", "issue_type", "column", "row_index", "value", "notes"]
    )
    print("No issue records generated.")

No issue records generated.


## 23. Export validation outputs

In [ ]:
OUTPUT_DIR = Path("/content/nexora_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_CSV = OUTPUT_DIR / "data_validation_report.csv"
SUMMARY_XLSX = OUTPUT_DIR / "data_validation_summary.xlsx"
ISSUES_CSV = OUTPUT_DIR / "data_quality_issues.csv"
READINESS_CSV = OUTPUT_DIR / "data_readiness_score.csv"
PROFILE_CSV = OUTPUT_DIR / "data_quality_profile.csv"

validation_report.to_csv(REPORT_CSV, index=False, encoding="utf-8")
data_quality_issues.to_csv(ISSUES_CSV, index=False, encoding="utf-8")
readiness_summary.to_csv(READINESS_CSV, index=False, encoding="utf-8")
quality_profile.to_csv(PROFILE_CSV, index=False, encoding="utf-8")

with pd.ExcelWriter(SUMMARY_XLSX, engine="openpyxl") as writer:
    readiness_summary.to_excel(writer, sheet_name="readiness_summary", index=False)
    validation_report.to_excel(writer, sheet_name="validation_report", index=False)
    failed_checks.to_excel(writer, sheet_name="failed_checks", index=False)
    missing_summary.to_excel(writer, sheet_name="missing_values", index=False)
    quality_profile.to_excel(writer, sheet_name="quality_profile", index=False)
    data_quality_issues.to_excel(writer, sheet_name="issue_samples", index=False)

print(f"Validation report : {REPORT_CSV}")
print(f"Summary workbook  : {SUMMARY_XLSX}")
print(f"Quality issues    : {ISSUES_CSV}")
print(f"Readiness score   : {READINESS_CSV}")
print(f"Quality profile   : {PROFILE_CSV}")

## 24. Go/no-go decision for the next project phase

Use the following rule:

- **READY**: proceed to MySQL staging-table creation.
- **READY WITH WARNINGS**: proceed only after documenting accepted warnings.
- **NEEDS IMPROVEMENT**: correct high-impact issues first.
- **NOT READY**: do not import into the data warehouse until critical failures are resolved.

In [21]:
print("=" * 60)
print("NEXORA DATA READINESS DECISION")
print("=" * 60)
print(f"Score  : {readiness_score:.2f}%")
print(f"Status : {readiness_status}")

if readiness_status == "READY":
    print("Decision: GO — dataset can proceed to MySQL staging.")
elif readiness_status == "READY WITH WARNINGS":
    print("Decision: CONDITIONAL GO — document and accept warnings first.")
elif readiness_status == "NEEDS IMPROVEMENT":
    print("Decision: HOLD — resolve failed high-impact checks.")
else:
    print("Decision: NO-GO — critical data-quality failures exist.")

NEXORA DATA READINESS DECISION
Score  : 100.00%
Status : READY
Decision: GO — dataset can proceed to MySQL staging.


## 25. Completion checklist

- [ ] Dataset contains approximately 180,519 rows.
- [ ] Required columns are present.
- [ ] `order_item_id` is unique and non-null.
- [ ] No exact duplicate rows remain.
- [ ] Order and shipping dates are valid.
- [ ] Shipping dates are not earlier than order dates.
- [ ] Sales, quantity, prices, and shipping durations follow valid ranges.
- [ ] Engineered features match their source formulas.
- [ ] Direct personally identifiable information has been removed.
- [ ] MySQL-readiness checks passed.
- [ ] Tableau-readiness warnings were reviewed.
- [ ] Readiness status is READY or READY WITH WARNINGS before continuing.